# V6.5 — External Staging Recovery + Compose QA (FULL FIX)

Notebook này sửa đúng case hiện tại:

```text
External stage candidates: []
```

mà **không reset các setting V6.5 đang dùng**.

Luồng của notebook:

```text
1. Tìm v6p5_ext_* trên Camber
       │
       ├─ có  -> reuse, KHÔNG rebuild dataset
       │
       └─ không -> chạy LOW-DISK external staging với đúng settings hiện tại
                    ↓
                  upload v6p5_ext_<hash>
                    ↓
                  tạo luôn 2 direct archives mà train notebook đang đòi
                    ↓
2. Compose QA
       ↓
3. v6p5_compose_<hash>
       ↓
4. Existing YOLO11n@640 data-ablation notebook có thể auto-discover
```

Notebook **không train model**. Nó chỉ đưa data pipeline về trạng thái hợp lệ trước khi train.

### Settings được giữ nguyên

- V6.2E: `v6p2e_23562a45b343`
- Open Images V7: `Balloon / Kite / Plastic bag / Bird / Person`
- `OI_MAX_SAMPLES_PER_CLASS = 250`
- `OI_SEED = 20261007`
- `SYNTH_PER_SUBTYPE = 100`
- `SYNTH_REQUIRE_MASK = True`
- Power Equipment: `dx_dg→broken_strand`, `yw→foreign_object`, `nw→bird_nest`, `dx_sg→review-only`
- `POWER_EQUIP_TRAIN_FRAC = 0.80`
- `RUN_PHASH = True`, Hamming ≤ 4
- parent pHash chỉ VAL/TEST
- không sửa TEST
- không auto-promotion

Có một setting mới duy nhất: `AUTO_STAGE_IF_MISSING=True`. Nó chỉ quyết định có tự chạy staging khi remote chưa có artifact hay không, không thay đổi dataset semantics.

In [ ]:
# ==============================================================================
# 0. CONFIG — GIỮ NGUYÊN V6.5 SETTINGS
# ==============================================================================
from pathlib import Path
import os, sys, re, io, json, math, hashlib, shutil, subprocess, zipfile, tarfile
import random, xml.etree.ElementTree as ET
import numpy as np
import pandas as pd

# ---------- Existing V6.5 settings ----------
NAMES = [
    'insulator',
    'insulator_broken',
    'insulator_flashover',
    'bird_nest',
    'foreign_object',
    'broken_strand',
]

V6P2E_ROOT = 'stash://bunpmc/projects/yolov11sdi/datasets'
V6P2E_TAG = 'v6p2e_23562a45b343'

OI_CLASSES = ['Balloon', 'Kite', 'Plastic bag', 'Bird', 'Person']
OI_MAX_SAMPLES_PER_CLASS = 250
OI_SEED = 20261007
SYNTH_PER_SUBTYPE = 100
SYNTH_REQUIRE_MASK = True

POWER_EQUIP_HF_REPO = 'sxiong/Power-equipment-image-dataset'
POWER_EQUIP_FILENAME = 'transmission line object detection.zip'
POWER_EQUIP_TRAIN_FRAC = 0.80
MPCD_GDRIVE_ID = '1KyciMmwL2_p_-mwckHFqG5fkBA1jzATv'

POWER_MAP = {'dx_dg': 5, 'yw': 4, 'nw': 3}
POWER_REVIEW_ONLY = {'dx_sg': 'strand_loosening_review_only'}

RUN_PHASH = True
PHASH_MAX_HAMMING = 4
PARENT_BG_POOL_MAX = 1200
PARENT_PHASH_SPLITS = ('val', 'test')
CLEAN_PARENT_ARCHIVES_AFTER_QA = True
MIN_FREE_GB_WARN = 3.0
MAX_CONTACTS_PER_SOURCE = 80
MAX_SYNTH_CONTACTS = 100

UPLOAD_CAMBER = True
STAGE_ROOT = 'stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage'
COMPOSE_ROOT = f'{STAGE_ROOT}/compose_qa'

RUN_SANITY_TRAIN = False
EXACT_BASE_DATA_YAML = None
SANITY_MODEL = 'yolo11n.pt'
SANITY_IMGSZ = 640
SANITY_EPOCHS = 10
SANITY_DEVICE = '0,1'

# ---------- Recovery / Compose controls ----------
AUTO_STAGE_IF_MISSING = True
STAGE_TAG_OVERRIDE = None

AUTO_APPROVE_RECOMMENDED = False
MANUAL_DECISIONS = {}
ALLOW_PHASH_CANDIDATES = False
ALLOW_EXACT_DUPLICATES = False
MAX_CONTACT_SHEETS = 24
ALLOW_EXISTING_COMPOSE_TAG = False

# ---------- Local paths ----------
W = Path('/kaggle/working')
ROOT = W / 'v6p5_external_expansion'
SRC = ROOT / 'sources'
BASE = ROOT / 'base_v6p2e'
OVERLAY = ROOT / 'v6p5_overlay'
QA = ROOT / 'qa'
CACHE = ROOT / 'cache'

COMPOSE_WORK = W / 'v6p5_compose_qa'
COMPOSE_CACHE = COMPOSE_WORK / 'cache'
COMPOSE_EXTRACT = COMPOSE_WORK / 'qa_extract'
COMPOSE_OUT = COMPOSE_WORK / 'out'

for p in [ROOT, SRC, BASE, OVERLAY, QA, CACHE,
          COMPOSE_WORK, COMPOSE_CACHE, COMPOSE_EXTRACT, COMPOSE_OUT]:
    p.mkdir(parents=True, exist_ok=True)

random.seed(OI_SEED)
np.random.seed(OI_SEED)

print('STAGE_ROOT:', STAGE_ROOT)
print('COMPOSE_ROOT:', COMPOSE_ROOT)
print('ROOT:', ROOT)

In [ ]:
# ==============================================================================
# 1. INSTALL + CAMBER AUTH
# ==============================================================================
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'ultralytics==8.4.172',
    'pyyaml', 'pillow', 'pandas', 'imagehash', 'tqdm',
    'gdown', 'huggingface_hub', 'fiftyone==1.22.1'
])

import yaml, imagehash
from PIL import Image, ImageDraw, ImageFont
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
from ultralytics import YOLO
import fiftyone.utils.openimages as fouo

key = os.environ.get('CAMBER_API_KEY')
if not key:
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret('CAMBER_API_KEY')
    except Exception:
        key = None

if not key:
    raise RuntimeError('Bật Kaggle Secret CAMBER_API_KEY trước.')

os.environ['CAMBER_API_KEY'] = key

camber_bin = Path.home() / '.camber/bin/camber'
if shutil.which('camber') is None and not camber_bin.exists():
    subprocess.check_call(
        'curl -sL https://cli.cambercloud.com/install-v2.sh | bash',
        shell=True,
    )

os.environ['PATH'] = (
    f'{Path.home()}/.camber/bin:{Path.home()}/.local/bin:'
    + os.environ.get('PATH', '')
)
CAMBER = shutil.which('camber') or str(camber_bin)
assert Path(CAMBER).exists(), 'Camber CLI not found'
print('CAMBER:', CAMBER)

In [ ]:
# ==============================================================================
# 2. HELPERS
# ==============================================================================
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}


def run(cmd, check=True):
    r = subprocess.run(
        cmd,
        text=True,
        capture_output=True,
        env=os.environ.copy(),
    )
    if check and r.returncode != 0:
        raise RuntimeError(
            f"Command failed ({r.returncode}): {' '.join(map(str, cmd))}\n"
            f"STDOUT:\n{r.stdout[:10000]}\n"
            f"STDERR:\n{r.stderr[:10000]}"
        )
    return r


def stash_ls(remote, recursive=True, check=True):
    cmd = [CAMBER, 'stash', 'ls']
    if recursive:
        cmd.append('-r')
    cmd.append(remote)
    return run(cmd, check=check)


def stash_cp(remote, local, reuse=True):
    local = Path(local)
    local.parent.mkdir(parents=True, exist_ok=True)
    if reuse and local.exists() and local.stat().st_size > 0:
        return local
    r = run([CAMBER, 'stash', 'cp', str(remote), str(local)], check=False)
    if r.returncode != 0 or not local.exists():
        raise RuntimeError(f'Camber download failed: {remote}\n{r.stderr or r.stdout}')
    return local


def stash_put(local, remote):
    local = Path(local)
    if not local.exists():
        raise FileNotFoundError(local)
    r = run([CAMBER, 'stash', 'cp', str(local), str(remote)], check=False)
    if r.returncode != 0:
        raise RuntimeError(f'Camber upload failed: {remote}\n{r.stderr or r.stdout}')


def sha256_file(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1024 * 1024), b''):
            h.update(b)
    return h.hexdigest()


def deterministic_fraction(key):
    h = hashlib.sha256(str(key).encode()).hexdigest()
    return int(h[:12], 16) / float(16 ** 12)


def stable_json_hash(obj):
    raw = json.dumps(
        obj,
        ensure_ascii=False,
        sort_keys=True,
        separators=(',', ':'),
    ).encode('utf-8')
    return hashlib.sha256(raw).hexdigest()


def safe_copy(src, dst):
    src = Path(src)
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return
    try:
        os.link(src, dst)
    except Exception:
        shutil.copy2(src, dst)


def disk_status(path=W):
    total, used, free = shutil.disk_usage(path)
    info = {
        'total_gb': total / 1024 ** 3,
        'used_gb': used / 1024 ** 3,
        'free_gb': free / 1024 ** 3,
    }
    print(
        f"Disk {path}: total={info['total_gb']:.2f} GB | "
        f"used={info['used_gb']:.2f} GB | free={info['free_gb']:.2f} GB"
    )
    if info['free_gb'] < MIN_FREE_GB_WARN:
        print(f"WARN: free space < {MIN_FREE_GB_WARN:.1f} GB")
    return info


def norm_names(names):
    if isinstance(names, dict):
        return [names[k] for k in sorted(names, key=lambda x: int(x))]
    return list(names or [])


def read_yolo_label(lp):
    out = []
    if not Path(lp).exists():
        return out
    for line in Path(lp).read_text(errors='ignore').splitlines():
        p = line.split()
        if len(p) < 5:
            continue
        try:
            out.append((int(float(p[0])), *map(float, p[1:5])))
        except Exception:
            pass
    return out


def write_yolo_label(lp, rows):
    Path(lp).parent.mkdir(parents=True, exist_ok=True)
    txt = '\n'.join(
        f'{int(c)} {x:.8f} {y:.8f} {w:.8f} {h:.8f}'
        for c, x, y, w, h in rows
        if w > 0 and h > 0
    )
    Path(lp).write_text(txt + ('\n' if txt else ''))


def yolo_xyxy(xc, yc, w, h):
    return [xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2]


def xyxy_to_yolo(b):
    x1, y1, x2, y2 = map(float, b)
    x1 = max(0, min(1, x1)); y1 = max(0, min(1, y1))
    x2 = max(0, min(1, x2)); y2 = max(0, min(1, y2))
    return [(x1 + x2) / 2, (y1 + y2) / 2, max(0, x2 - x1), max(0, y2 - y1)]


def make_contact(image_path, boxes, labels, out_path, title=''):
    im = Image.open(image_path).convert('RGB')
    scale = min(1.0, 1200 / max(im.size))
    if scale < 1:
        im = im.resize((int(im.width * scale), int(im.height * scale)))
    W0, H0 = im.size
    head = 55
    canvas = Image.new('RGB', (W0, H0 + head), 'white')
    canvas.paste(im, (0, head))
    d = ImageDraw.Draw(canvas)
    try:
        font = ImageFont.truetype('DejaVuSans.ttf', 16)
    except Exception:
        font = ImageFont.load_default()
    d.text((8, 8), title[:160], fill='black', font=font)
    for box, label in zip(boxes, labels):
        x1, y1, x2, y2 = box
        rect = [x1 * W0, y1 * H0 + head, x2 * W0, y2 * H0 + head]
        d.rectangle(rect, outline='red', width=4)
        d.text((rect[0] + 2, max(head, rect[1] - 18)), str(label), fill='red', font=font)
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    canvas.save(out_path, quality=90)


def read_csv_safe(path):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()


def normalize_decision(x):
    x = str(x or '').strip().upper()
    return x if x in {'KEEP', 'DROP'} else ''


def extract_tag_tokens(text, prefix):
    pat = rf'{re.escape(prefix)}[0-9a-fA-F]{{12}}'
    return sorted(set(re.findall(pat, text or '')))


def dir_digest(root):
    items = []
    for p in sorted(Path(root).rglob('*')):
        if p.is_file():
            items.append(f'{p.relative_to(root)}:{sha256_file(p)}')
    return hashlib.sha256('\n'.join(items).encode()).hexdigest()


def tar_find_member(member_names, suffix):
    suffix = str(suffix).replace('\\', '/').lstrip('/')
    hits = [n for n in member_names if n.replace('\\', '/').endswith(suffix)]
    if len(hits) == 1:
        return hits[0]
    if len(hits) == 0:
        return None
    raise RuntimeError(f'Ambiguous TAR member suffix {suffix}: {hits[:10]}')


def add_bytes_to_tar(out_tar, arcname, raw, src_info=None):
    info = tarfile.TarInfo(name=str(arcname).replace('\\', '/'))
    info.size = len(raw)
    info.mode = getattr(src_info, 'mode', 0o644) or 0o644
    info.mtime = getattr(src_info, 'mtime', 0) or 0
    out_tar.addfile(info, io.BytesIO(raw))

print('Helpers ready.')

In [ ]:
# ==============================================================================
# 3. DISCOVER EXISTING EXTERNAL STAGE — ROBUST + READ ONLY
# ==============================================================================
def discover_stage_tags():
    texts = []
    for recursive in (False, True):
        r = stash_ls(STAGE_ROOT, recursive=recursive, check=False)
        if r.returncode == 0:
            texts.append(r.stdout or '')
            texts.append(r.stderr or '')
    joined = '\n'.join(texts)
    return extract_tag_tokens(joined, 'v6p5_ext_'), joined

if STAGE_TAG_OVERRIDE:
    STAGE_TAG = STAGE_TAG_OVERRIDE.strip()
    STAGE_EXISTS = True
    listing_raw = ''
else:
    stage_candidates, listing_raw = discover_stage_tags()
    print('External stage candidates:', stage_candidates)
    if len(stage_candidates) == 1:
        STAGE_TAG = stage_candidates[0]
        STAGE_EXISTS = True
    elif len(stage_candidates) > 1:
        raise RuntimeError(
            'Có nhiều v6p5_ext_* candidates. Set STAGE_TAG_OVERRIDE bằng tag cần dùng:\n'
            + '\n'.join(stage_candidates)
        )
    else:
        STAGE_TAG = None
        STAGE_EXISTS = False

print('STAGE_EXISTS:', STAGE_EXISTS)
if STAGE_EXISTS:
    print('SELECTED STAGE_TAG:', STAGE_TAG)
else:
    print('Không có external stage remote. Recovery staging sẽ chạy:', AUTO_STAGE_IF_MISSING)

## Recovery staging

Các cell tiếp theo **chỉ thực sự làm việc khi `STAGE_EXISTS == False`**.

Nếu Camber đã có đúng một `v6p5_ext_*`, chúng chỉ in `SKIP` và không rebuild dataset. Nhờ vậy rerun notebook không tự nhiên tải lại nửa thế giới chỉ vì một cell phía sau cần chạy lại. Một tiến bộ đáng kể đối với loài notebook.

In [ ]:
# ==============================================================================
# 4. RECOVERY: STREAM V6.2E PARENT + BUILD BOUNDED TRAIN BACKGROUND POOL
# ==============================================================================
if not STAGE_EXISTS:
    if not AUTO_STAGE_IF_MISSING:
        raise RuntimeError(
            'Không có v6p5_ext_* và AUTO_STAGE_IF_MISSING=False. '
            'Bật True để notebook tự recovery staging.'
        )

    PARENT_CACHE = BASE / 'cache'
    PARENT_CACHE.mkdir(parents=True, exist_ok=True)

    stale_dirs = [
        PARENT_CACHE / 'base_extract',
        PARENT_CACHE / 'overlay_extract',
        PARENT_CACHE / 'reconciled',
        BASE / 'runtime',
    ]
    for p in stale_dirs:
        if p.exists():
            print('Removing stale full extraction:', p)
            shutil.rmtree(p, ignore_errors=True)

    disk_status()

    freeze_path_parent = PARENT_CACHE / 'v6p2e.freeze.json'
    stash_cp(
        f'{V6P2E_ROOT}/v6p2e/{V6P2E_TAG}/v6p2e.freeze.json',
        freeze_path_parent,
    )
    parent_freeze = json.loads(freeze_path_parent.read_text())

    base_tar_parent = PARENT_CACHE / f"base_{parent_freeze['base_tag']}.tar"
    overlay_tar_parent = PARENT_CACHE / f"overlay_{parent_freeze['overlay_tag']}.tar.gz"
    recon_zip_parent = PARENT_CACHE / 'v6p2e_reconciled_labels.zip'
    manifest_csv_parent = PARENT_CACHE / 'v6p2e_split_manifest.csv'

    stash_cp(
        f"{V6P2E_ROOT}/curated_v6_rs1280_{parent_freeze['base_tag']}.tar",
        base_tar_parent,
    )
    stash_cp(
        f"{V6P2E_ROOT}/v6p1/{parent_freeze['parent_dataset_tag']}/"
        f"v6p1_fotl_overlay_{parent_freeze['overlay_tag']}.tar.gz",
        overlay_tar_parent,
    )
    stash_cp(
        f'{V6P2E_ROOT}/v6p2e/{V6P2E_TAG}/v6p2e_reconciled_labels.zip',
        recon_zip_parent,
    )
    stash_cp(
        f'{V6P2E_ROOT}/v6p2e/{V6P2E_TAG}/v6p2e_split_manifest.csv',
        manifest_csv_parent,
    )

    manifest = pd.read_csv(manifest_csv_parent)
    print('V6.2E manifest rows:', len(manifest))
    print(manifest['split'].value_counts())

    BASE_TAR_H = tarfile.open(base_tar_parent, 'r:*')
    OVERLAY_TAR_H = tarfile.open(overlay_tar_parent, 'r:*')
    RECON_ZIP_H = zipfile.ZipFile(recon_zip_parent, 'r')

    def build_tar_suffix_index(tf):
        idx = {}
        duplicate_keys = set()
        for m in tqdm(tf.getmembers(), desc=f'index {Path(tf.name).name}'):
            if not m.isfile():
                continue
            n = m.name.replace('\\', '/').lstrip('./')
            key = None
            pi = n.find('images/')
            pl = n.find('labels/')
            if pi >= 0:
                key = n[pi:]
            elif pl >= 0:
                key = n[pl:]
            if key is None:
                continue
            if key in idx:
                duplicate_keys.add(key)
            else:
                idx[key] = m
        if duplicate_keys:
            print('WARN duplicate archive suffix keys:', len(duplicate_keys))
        return idx

    BASE_IDX = build_tar_suffix_index(BASE_TAR_H)
    OVERLAY_IDX = build_tar_suffix_index(OVERLAY_TAR_H)
    RECON_IDX = {
        Path(n).name: n
        for n in RECON_ZIP_H.namelist()
        if n.endswith('.txt')
    }

    def row_source_keys(row):
        old_path = str(row['image_path_source']).replace('\\', '/')
        old_split = str(row['previous_v6p1_split'])
        marker = f'/images/{old_split}/'
        if marker not in old_path:
            raise RuntimeError(f'Bad manifest source path: {old_path}')
        suffix = old_path.split(marker, 1)[1]
        if suffix.startswith('base/'):
            source = 'base'; rel = suffix[len('base/'):]
        elif suffix.startswith('fotl/'):
            source = 'fotl'; rel = suffix[len('fotl/'):]
        else:
            raise RuntimeError(f'Unknown provenance suffix: {suffix}')
        rel = rel.replace('\\', '/')
        img_key = f'images/{old_split}/{rel}'
        label_rel = str(Path(rel).with_suffix('.txt')).replace('\\', '/')
        lbl_key = f'labels/{old_split}/{label_rel}'
        return source, rel, img_key, lbl_key

    def source_handles(source):
        if source == 'base':
            return BASE_TAR_H, BASE_IDX
        if source == 'fotl':
            return OVERLAY_TAR_H, OVERLAY_IDX
        raise KeyError(source)

    def read_parent_image_bytes(row):
        source, rel, img_key, _ = row_source_keys(row)
        tf, idx = source_handles(source)
        member = idx.get(img_key)
        if member is None:
            raise KeyError(f'Missing archive image member: {source}:{img_key}')
        f = tf.extractfile(member)
        if f is None:
            raise RuntimeError(f'Cannot extract image member: {member.name}')
        return f.read()

    def read_parent_label_text(row):
        sha_name = f"{row['sha256']}.txt"
        if str(row.get('resolution', '')).startswith('merged'):
            zn = RECON_IDX.get(sha_name)
            if zn is not None:
                return RECON_ZIP_H.read(zn).decode('utf-8', errors='ignore')
        source, rel, _, lbl_key = row_source_keys(row)
        tf, idx = source_handles(source)
        member = idx.get(lbl_key)
        if member is None:
            raise KeyError(f'Missing archive label member: {source}:{lbl_key}')
        f = tf.extractfile(member)
        if f is None:
            raise RuntimeError(f'Cannot extract label member: {member.name}')
        return f.read().decode('utf-8', errors='ignore')

    def parse_yolo_text(txt):
        rows = []
        for line in str(txt).splitlines():
            p = line.strip().split()
            if len(p) < 5:
                continue
            try:
                rows.append((int(float(p[0])), *map(float, p[1:5])))
            except Exception:
                continue
        return rows

    BG_POOL = BASE / 'train_background_pool'
    BG_IMG = BG_POOL / 'images'
    BG_LBL = BG_POOL / 'labels'
    if BG_POOL.exists():
        shutil.rmtree(BG_POOL)
    BG_IMG.mkdir(parents=True, exist_ok=True)
    BG_LBL.mkdir(parents=True, exist_ok=True)

    train_manifest = manifest[manifest['split'].astype(str) == 'train'].copy()
    train_manifest['_sort'] = train_manifest['sha256'].astype(str).map(
        lambda s: hashlib.sha256((s + str(OI_SEED)).encode()).hexdigest()
    )
    train_manifest = train_manifest.sort_values('_sort')

    bg_rows, fail_rows = [], []
    for idx, row in tqdm(
        train_manifest.iterrows(),
        total=len(train_manifest),
        desc='select TRAIN power-line backgrounds',
    ):
        if len(bg_rows) >= PARENT_BG_POOL_MAX:
            break
        try:
            label_txt = read_parent_label_text(row)
            labels = parse_yolo_text(label_txt)
            if not any(cid == 0 for cid, *_ in labels):
                continue
            img_bytes = read_parent_image_bytes(row)
            _, rel, _, _ = row_source_keys(row)
            suffix = Path(rel).suffix.lower() or '.jpg'
            name = f"{str(row['sha256'])[:16]}_{idx:06d}{suffix}"
            ip = BG_IMG / name
            lp = BG_LBL / Path(name).with_suffix('.txt')
            ip.write_bytes(img_bytes)
            lp.write_text(label_txt)
            bg_rows.append({
                'manifest_index': int(idx),
                'sha256': str(row['sha256']),
                'split': 'train',
                'image': str(ip),
                'label': str(lp),
                'source_image_path': str(row['image_path_source']),
            })
        except Exception as e:
            fail_rows.append({
                'manifest_index': int(idx),
                'sha256': str(row.get('sha256', '')),
                'error': repr(e),
            })

    df_bg_pool = pd.DataFrame(bg_rows)
    df_bg_pool.to_csv(QA / 'parent_train_background_pool.csv', index=False)
    if fail_rows:
        pd.DataFrame(fail_rows).to_csv(QA / 'parent_stream_failures.csv', index=False)

    print('Materialized TRAIN backgrounds:', len(df_bg_pool))
    assert len(df_bg_pool) >= 100, f'Too few structural backgrounds: {len(df_bg_pool)}'
    disk_status()
else:
    print('SKIP recovery parent stream: existing external stage found.')

In [ ]:
# ==============================================================================
# 5. RECOVERY: OPEN IMAGES V7 + OBJECT CROPS
# ==============================================================================
if not STAGE_EXISTS:
    OI_ROOT = SRC / 'open_images_v7'
    OI_ROOT.mkdir(parents=True, exist_ok=True)

    avail = set(fouo.get_classes(version='v7'))
    seg_avail = set(fouo.get_segmentation_classes(version='v7'))

    for cls in OI_CLASSES:
        assert cls in avail, f'Open Images class missing: {cls}'
        types = ['detections'] + (['segmentations'] if cls in seg_avail else [])
        print('OpenImages', cls, 'seg=', cls in seg_avail)
        fouo.download_open_images_split(
            dataset_dir=str(OI_ROOT),
            split='train',
            version='v7',
            label_types=types,
            classes=[cls],
            shuffle=True,
            seed=OI_SEED,
            max_samples=OI_MAX_SAMPLES_PER_CLASS,
            num_workers=8,
        )

    OI_CROPS = SRC / 'oi_object_crops'
    OI_CROPS.mkdir(parents=True, exist_ok=True)
    oi_rows = []

    for cls in OI_CLASSES:
        types = ['detections'] + (['segmentations'] if cls in seg_avail else [])
        imp = fouo.OpenImagesV7DatasetImporter(
            dataset_dir=str(OI_ROOT),
            label_types=types,
            classes=[cls],
            only_matching=True,
            shuffle=False,
        )
        imp.setup()
        count = 0
        try:
            for sample in imp:
                image_path, metadata, labels = sample
                image_path = Path(image_path)
                if not image_path.exists():
                    continue
                if not isinstance(labels, dict):
                    labels = {'detections': labels}

                cand = []
                segs = labels.get('segmentations')
                if segs is not None and hasattr(segs, 'detections'):
                    cand = [('segmentation', d) for d in segs.detections if d.label == cls]

                if not cand:
                    dets = labels.get('detections')
                    if dets is not None and hasattr(dets, 'detections'):
                        cand = [('detection', d) for d in dets.detections if d.label == cls]

                if not cand:
                    continue

                im = Image.open(image_path).convert('RGB')
                W0, H0 = im.size

                for j, (kind, det) in enumerate(cand):
                    x, y, w, h = map(float, det.bounding_box)
                    x1 = max(0, int(x * W0)); y1 = max(0, int(y * H0))
                    x2 = min(W0, int((x + w) * W0)); y2 = min(H0, int((y + h) * H0))
                    if x2 - x1 < 8 or y2 - y1 < 8:
                        continue

                    crop = im.crop((x1, y1, x2, y2))
                    mask_arr = None
                    if kind == 'segmentation':
                        if getattr(det, 'mask', None) is not None:
                            mask_arr = np.asarray(det.mask)
                        elif getattr(det, 'mask_path', None) and Path(det.mask_path).exists():
                            mask_arr = np.asarray(Image.open(det.mask_path).convert('L'))

                    mask_ok = mask_arr is not None
                    rgba = crop.convert('RGBA')
                    if mask_ok:
                        mask = Image.fromarray((mask_arr > 0).astype(np.uint8) * 255)
                        mask = mask.resize(crop.size, Image.Resampling.NEAREST)
                        rgba.putalpha(mask)
                    else:
                        rgba.putalpha(Image.new('L', crop.size, 255))

                    subtype = cls.lower().replace(' ', '_')
                    uid = hashlib.sha256(f'{image_path}|{cls}|{j}'.encode()).hexdigest()[:16]
                    op = OI_CROPS / subtype / f'{uid}.png'
                    op.parent.mkdir(parents=True, exist_ok=True)
                    rgba.save(op)

                    oi_rows.append({
                        'source': 'open_images_v7',
                        'source_split': 'train',
                        'source_image': str(image_path),
                        'source_sha256': sha256_file(image_path),
                        'foreign_subtype': subtype,
                        'crop_path': str(op),
                        'mask_available': mask_ok,
                    })
                    count += 1
        finally:
            imp.close()
        print(cls, 'crops=', count)

    df_oi = pd.DataFrame(oi_rows)
    df_oi.to_csv(QA / 'open_images_crop_manifest.csv', index=False)
    if len(df_oi):
        display(df_oi.groupby(['foreign_subtype', 'mask_available']).size().unstack(fill_value=0))
else:
    print('SKIP Open Images: existing external stage found.')

In [ ]:
# ==============================================================================
# 6. RECOVERY: MPCD — DISK AWARE
# ==============================================================================
if not STAGE_EXISTS:
    import gdown

    MPCD_ZIP = SRC / 'MPCD.zip'
    MPCD_DIR = SRC / 'MPCD'
    mpcd_ready = MPCD_DIR.exists() and any(MPCD_DIR.rglob('data.yaml'))

    if not mpcd_ready:
        if MPCD_DIR.exists():
            shutil.rmtree(MPCD_DIR, ignore_errors=True)
        if not MPCD_ZIP.exists():
            url = f'https://drive.google.com/uc?id={MPCD_GDRIVE_ID}'
            print('Downloading MPCD...')
            result = gdown.download(url, str(MPCD_ZIP), quiet=False)
            if result is None:
                raise RuntimeError(
                    'MPCD Google Drive download failed. '
                    f'Place archive manually at {MPCD_ZIP}'
                )
        MPCD_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(MPCD_ZIP) as z:
            z.extractall(MPCD_DIR)
        MPCD_ZIP.unlink(missing_ok=True)

    disk_status()

    def resolve_yolo_dataset_yaml(yp):
        cfg = yaml.safe_load(Path(yp).read_text())
        base = Path(cfg.get('path', Path(yp).parent))
        if not base.is_absolute():
            base = (Path(yp).parent / base).resolve()
        return cfg, base, norm_names(cfg.get('names', []))

    def image_to_label_path(ip):
        s = str(ip).replace('\\', '/')
        if '/images/' in s:
            return Path(s.replace('/images/', '/labels/')).with_suffix('.txt')
        return None

    mpcd_records = []
    for yp in MPCD_DIR.rglob('data.yaml'):
        try:
            cfg, base, src_names = resolve_yolo_dataset_yaml(yp)
        except Exception:
            continue

        broken_ids = []
        for cid, name in enumerate(src_names):
            n = str(name).lower().replace('_', ' ').replace('-', ' ')
            if any(k in n for k in ['broken', 'fracture', 'fractured', 'discontinu', 'strand break']):
                broken_ids.append(cid)
        if not broken_ids:
            continue

        print('MPCD detection yaml:', yp)
        print('names:', src_names)
        print('mapped broken ids:', broken_ids)

        for split_key in ['train', 'val', 'test']:
            raw = cfg.get(split_key)
            if raw is None:
                continue
            p = Path(raw)
            if not p.is_absolute():
                p = base / p

            if p.is_file():
                image_paths = []
                for x in p.read_text().splitlines():
                    if not x.strip():
                        continue
                    q = Path(x.strip())
                    image_paths.append(q if q.is_absolute() else base / q)
            elif p.exists():
                image_paths = [x for x in p.rglob('*') if x.suffix.lower() in IMG_EXTS]
            else:
                continue

            for ip in image_paths:
                lp = image_to_label_path(ip)
                if lp is None or not lp.exists():
                    continue
                mapped = []
                for cid, xc, yc, w, h in read_yolo_label(lp):
                    if cid in broken_ids:
                        mapped.append((5, xc, yc, w, h))
                if not mapped:
                    continue
                mpcd_records.append({
                    'source': 'MPCD',
                    'source_split': split_key,
                    'source_image': str(ip),
                    'source_label': str(lp),
                    'mapped_rows': mapped,
                    'source_names': src_names,
                    'license': 'AGPL-3.0 project repository; verify dataset-source terms before redistribution',
                })

    uniq = {}
    for r in mpcd_records:
        uniq[(r['source_split'], r['source_image'])] = r
    mpcd_records = list(uniq.values())

    print('MPCD mapped images:', len(mpcd_records))
    if mpcd_records:
        print(pd.Series([r['source_split'] for r in mpcd_records]).value_counts())
    else:
        raise RuntimeError('MPCD parsed but produced 0 mapped broken-strand images.')
else:
    print('SKIP MPCD: existing external stage found.')

In [ ]:
# ==============================================================================
# 7. RECOVERY: POWER EQUIPMENT — DISK AWARE
# ==============================================================================
if not STAGE_EXISTS:
    PE_DIR = SRC / 'power_equipment_object_detection'
    HF_TMP = CACHE / 'hf_power_equipment'

    pe_ready = PE_DIR.exists() and (
        any(PE_DIR.rglob('*.xml')) or
        any(p.suffix.lower() in IMG_EXTS for p in PE_DIR.rglob('*'))
    )

    if not pe_ready:
        if PE_DIR.exists():
            shutil.rmtree(PE_DIR, ignore_errors=True)
        if HF_TMP.exists():
            shutil.rmtree(HF_TMP, ignore_errors=True)

        downloaded_path = hf_hub_download(
            repo_id=POWER_EQUIP_HF_REPO,
            filename=POWER_EQUIP_FILENAME,
            repo_type='dataset',
            cache_dir=str(HF_TMP),
        )
        downloaded_path = Path(downloaded_path)
        PE_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(downloaded_path) as z:
            z.extractall(PE_DIR)
        shutil.rmtree(HF_TMP, ignore_errors=True)

    disk_status()

    pe_records = []
    pe_review = []
    all_images_by_stem = {}
    for ip in PE_DIR.rglob('*'):
        if ip.suffix.lower() in IMG_EXTS:
            all_images_by_stem.setdefault(ip.stem, []).append(ip)

    xmls = list(PE_DIR.rglob('*.xml'))
    print('PowerEquipment XML files:', len(xmls))

    for xp in tqdm(xmls, desc='parse PowerEquipment XML'):
        try:
            root_xml = ET.parse(xp).getroot()
        except Exception:
            continue

        filename = root_xml.findtext('filename')
        candidate = None
        if filename:
            q = xp.parent / filename
            if q.exists():
                candidate = q
            else:
                hits = all_images_by_stem.get(Path(filename).stem, [])
                candidate = hits[0] if hits else None
        else:
            hits = all_images_by_stem.get(xp.stem, [])
            candidate = hits[0] if hits else None

        if candidate is None or not Path(candidate).exists():
            continue

        size = root_xml.find('size')
        if size is None:
            with Image.open(candidate) as im:
                W0, H0 = im.size
        else:
            W0 = int(float(size.findtext('width')))
            H0 = int(float(size.findtext('height')))

        mapped_rows = []
        review_rows = []
        for obj in root_xml.findall('object'):
            name = (obj.findtext('name') or '').strip()
            bb = obj.find('bndbox')
            if bb is None:
                continue
            xmin = float(bb.findtext('xmin')); ymin = float(bb.findtext('ymin'))
            xmax = float(bb.findtext('xmax')); ymax = float(bb.findtext('ymax'))
            box = [
                max(0.0, min(1.0, xmin / W0)),
                max(0.0, min(1.0, ymin / H0)),
                max(0.0, min(1.0, xmax / W0)),
                max(0.0, min(1.0, ymax / H0)),
            ]

            if name in POWER_MAP:
                xc, yc, w, h = xyxy_to_yolo(box)
                mapped_rows.append((POWER_MAP[name], xc, yc, w, h))

            if name in POWER_REVIEW_ONLY:
                review_rows.append({
                    'source': 'PowerEquipment',
                    'source_image': str(candidate),
                    'source_xml': str(xp),
                    'source_class': name,
                    'review_role': POWER_REVIEW_ONLY[name],
                    'bbox_xyxy': box,
                })

        if mapped_rows:
            split = (
                'train'
                if deterministic_fraction(str(candidate)) < POWER_EQUIP_TRAIN_FRAC
                else 'external_eval'
            )
            pe_records.append({
                'source': 'PowerEquipment',
                'source_split': split,
                'source_image': str(candidate),
                'source_label': str(xp),
                'mapped_rows': mapped_rows,
                'license': 'MIT repository; verify original data redistribution terms',
            })
        pe_review.extend(review_rows)

    df_pe_review = pd.DataFrame(pe_review)
    df_pe_review.to_csv(QA / 'power_equipment_review_only_dx_sg.csv', index=False)

    print('PowerEquipment mapped images:', len(pe_records))
    if pe_records:
        print(pd.Series([r['source_split'] for r in pe_records]).value_counts())
    print('dx_sg review-only boxes:', len(df_pe_review))

    if len(pe_records) == 0:
        raise RuntimeError('PowerEquipment parsed but produced 0 mapped candidate images.')
else:
    print('SKIP PowerEquipment: existing external stage found.')

In [ ]:
# ==============================================================================
# 8. RECOVERY: BUILD DIRECT OVERLAY + SOURCE-SPECIFIC TRAIN ARCHIVES
# ==============================================================================
if not STAGE_EXISTS:
    # Rebuild local direct overlay deterministically.
    direct_root = OVERLAY / 'direct'
    if direct_root.exists():
        shutil.rmtree(direct_root)

    for s in ['train', 'external_eval']:
        (OVERLAY / 'direct/images' / s).mkdir(parents=True, exist_ok=True)
        (OVERLAY / 'direct/labels' / s).mkdir(parents=True, exist_ok=True)

    direct = []

    def add_direct(rec, target):
        ip = Path(rec['source_image'])
        sha = sha256_file(ip)
        name = f"{rec['source'].lower()}_{sha[:16]}{ip.suffix.lower()}"
        di = OVERLAY / 'direct/images' / target / name
        dl = OVERLAY / 'direct/labels' / target / Path(name).with_suffix('.txt')
        safe_copy(ip, di)
        write_yolo_label(dl, rec['mapped_rows'])
        classes = sorted({int(x[0]) for x in rec['mapped_rows']})
        direct.append({
            'overlay_image': str(di),
            'overlay_label': str(dl),
            'target_split': target,
            'source': rec['source'],
            'source_split': rec['source_split'],
            'source_image': str(ip),
            'source_sha256': sha,
            'mapped_classes': ','.join(map(str, classes)),
            'mapped_class_names': ','.join(NAMES[c] for c in classes),
            'n_mapped_boxes': len(rec['mapped_rows']),
            'license_note': rec.get('license', ''),
        })

    for r in mpcd_records:
        add_direct(r, 'train' if r['source_split'] == 'train' else 'external_eval')
    for r in pe_records:
        add_direct(r, r['source_split'])

    df_direct = pd.DataFrame(direct)
    df_direct.to_csv(QA / 'direct_overlay_manifest.csv', index=False)
    display(df_direct.groupby(['source', 'target_split', 'mapped_class_names']).size().reset_index(name='images'))

    # --------------------------------------------------------------------------
    # FIX: train notebook expects these two exact fixed remote archives.
    # The old staging notebook did not create them, so create them here.
    # --------------------------------------------------------------------------
    DIRECT_STAGE_DIR = ROOT / 'direct_stage_archives'
    DIRECT_STAGE_DIR.mkdir(parents=True, exist_ok=True)

    MPCD_STAGE_ROOT = ROOT / 'mpcd_broken_strand'
    PE_STAGE_ROOT = ROOT / 'power_equipment_stage'
    for p in [MPCD_STAGE_ROOT, PE_STAGE_ROOT]:
        if p.exists():
            shutil.rmtree(p)
        for s in ['train', 'external_eval']:
            (p / 'images' / s).mkdir(parents=True, exist_ok=True)
            (p / 'labels' / s).mkdir(parents=True, exist_ok=True)

    for _, r in df_direct.iterrows():
        src = str(r['source'])
        split = str(r['target_split'])
        ip = Path(r['overlay_image'])
        lp = Path(r['overlay_label'])
        root = MPCD_STAGE_ROOT if src == 'MPCD' else PE_STAGE_ROOT
        safe_copy(ip, root / 'images' / split / ip.name)
        safe_copy(lp, root / 'labels' / split / lp.name)

    mpcd_stage_tar = DIRECT_STAGE_DIR / 'mpcd_broken_strand_stage.tar.gz'
    pe_stage_tar = DIRECT_STAGE_DIR / 'power_equipment_stage.tar.gz'

    for p in [mpcd_stage_tar, pe_stage_tar]:
        p.unlink(missing_ok=True)

    with tarfile.open(mpcd_stage_tar, 'w:gz') as t:
        t.add(MPCD_STAGE_ROOT, arcname='mpcd_broken_strand')
    with tarfile.open(pe_stage_tar, 'w:gz') as t:
        t.add(PE_STAGE_ROOT, arcname='power_equipment_stage')

    print('MPCD stage archive:', mpcd_stage_tar, mpcd_stage_tar.stat().st_size / 1024**2, 'MB')
    print('PowerEquipment stage archive:', pe_stage_tar, pe_stage_tar.stat().st_size / 1024**2, 'MB')
else:
    print('SKIP direct overlay rebuild: existing external stage found.')

In [ ]:
# ==============================================================================
# 9. RECOVERY: STRUCTURAL ANCHORS + RELATION-AWARE SYNTHETIC FOREIGN OBJECTS
# ==============================================================================
if not STAGE_EXISTS:
    anchors = []
    for _, r in tqdm(df_bg_pool.iterrows(), total=len(df_bg_pool), desc='find insulator anchors'):
        ip = Path(r['image'])
        lp = Path(r['label'])
        for j, (cid, xc, yc, w, h) in enumerate(read_yolo_label(lp)):
            if cid == 0:
                anchors.append({
                    'image': str(ip),
                    'label': str(lp),
                    'background_sha256': r['sha256'],
                    'anchor_index': j,
                    'anchor_xc': xc,
                    'anchor_yc': yc,
                    'anchor_w': w,
                    'anchor_h': h,
                })

    df_anchor = pd.DataFrame(anchors)
    print('anchors=', len(df_anchor), 'unique backgrounds=', df_anchor['image'].nunique())
    assert len(df_anchor) > 0, 'No insulator anchors found in TRAIN background pool.'

    SYN = OVERLAY / 'synthetic_foreign'
    if SYN.exists():
        shutil.rmtree(SYN)
    (SYN / 'images/train').mkdir(parents=True, exist_ok=True)
    (SYN / 'labels/train').mkdir(parents=True, exist_ok=True)

    def resize_rgba(obj, longside):
        ow, oh = obj.size
        sc = longside / max(ow, oh)
        return obj.resize(
            (max(2, int(ow * sc)), max(2, int(oh * sc))),
            Image.Resampling.LANCZOS,
        )

    def compose(bg, obj, a, subtype, rng):
        W0, H0 = bg.size
        ax, ay, aw, ah = a
        acx, acy = ax * W0, ay * H0
        along = max(aw * W0, ah * H0)

        if subtype == 'person':
            rel = rng.uniform(.45, .90)
        elif subtype in {'balloon', 'kite'}:
            rel = rng.uniform(.20, .55)
        elif subtype == 'bird':
            rel = rng.uniform(.10, .28)
        else:
            rel = rng.uniform(.12, .35)

        obj = resize_rgba(
            obj,
            max(12, min(int(along * rel), int(min(W0, H0) * .30))),
        )

        if subtype == 'person':
            dx = rng.uniform(-.8, .8) * max(aw * W0, 40)
            dy = rng.uniform(.35, 1.2) * max(ah * H0, 40)
        else:
            dx = rng.uniform(-.7, .7) * max(aw * W0, 35)
            dy = rng.uniform(-.6, .6) * max(ah * H0, 35)

        x1 = int(acx + dx - obj.width / 2)
        y1 = int(acy + dy - obj.height / 2)
        x1 = max(0, min(W0 - obj.width, x1))
        y1 = max(0, min(H0 - obj.height, y1))
        x2 = x1 + obj.width
        y2 = y1 + obj.height
        if x2 <= x1 or y2 <= y1:
            return None

        out = bg.convert('RGBA')
        out.alpha_composite(obj, (x1, y1))
        return out.convert('RGB'), [x1 / W0, y1 / H0, x2 / W0, y2 / H0]

    usable = (
        df_oi[df_oi['mask_available'] == True].copy()
        if SYNTH_REQUIRE_MASK
        else df_oi.copy()
    )
    if len(usable) == 0:
        raise RuntimeError('No usable OpenImages object crops for synthetic generation.')

    rng = random.Random(OI_SEED)
    syn = []

    for subtype in sorted(usable['foreign_subtype'].unique()):
        pool = usable[usable['foreign_subtype'] == subtype].to_dict('records')
        made = 0
        attempts = 0
        while made < SYNTH_PER_SUBTYPE and attempts < SYNTH_PER_SUBTYPE * 20:
            attempts += 1
            objrec = rng.choice(pool)
            a = df_anchor.sample(1, random_state=rng.randint(0, 10**9)).iloc[0]
            try:
                bg = Image.open(a['image']).convert('RGB')
                obj = Image.open(objrec['crop_path']).convert('RGBA')
            except Exception:
                continue

            res = compose(
                bg,
                obj,
                (
                    float(a['anchor_xc']),
                    float(a['anchor_yc']),
                    float(a['anchor_w']),
                    float(a['anchor_h']),
                ),
                subtype,
                rng,
            )
            if res is None:
                continue

            sim, box = res
            rows = read_yolo_label(a['label'])
            fx, fy, fw, fh = xyxy_to_yolo(box)
            rows.append((4, fx, fy, fw, fh))

            uid = hashlib.sha256(
                f"{a['image']}|{objrec['crop_path']}|{made}|{OI_SEED}".encode()
            ).hexdigest()[:18]

            di = SYN / 'images/train' / f'syn_{subtype}_{uid}.jpg'
            dl = SYN / 'labels/train' / f'syn_{subtype}_{uid}.txt'
            sim.save(di, quality=92)
            write_yolo_label(dl, rows)

            syn.append({
                'overlay_image': str(di),
                'overlay_label': str(dl),
                'source': 'OpenImagesV7_synthetic_relation',
                'foreign_subtype': subtype,
                'oi_source_image': objrec['source_image'],
                'oi_source_sha256': objrec['source_sha256'],
                'oi_crop': objrec['crop_path'],
                'background_image': a['image'],
                'background_sha256': sha256_file(a['image']),
                'anchor_class': 'insulator',
                'foreign_box_xyxy': box,
                'synthetic': True,
                'train_only': True,
            })
            made += 1

        print(subtype, 'made=', made)

    df_synth = pd.DataFrame(syn)
    df_synth.to_csv(QA / 'synthetic_foreign_manifest.csv', index=False)
    display(df_synth['foreign_subtype'].value_counts())
else:
    print('SKIP synthetic rebuild: existing external stage found.')

In [ ]:
# ==============================================================================
# 10. RECOVERY: CONTACT SHEETS + DEDUP QA
# ==============================================================================
if not STAGE_EXISTS:
    CONTACT = QA / 'contacts'
    CONTACT.mkdir(parents=True, exist_ok=True)

    for source in df_direct['source'].unique():
        for _, r in df_direct[df_direct['source'] == source].head(MAX_CONTACTS_PER_SOURCE).iterrows():
            boxes, labs = [], []
            for cid, xc, yc, w, h in read_yolo_label(r['overlay_label']):
                boxes.append(yolo_xyxy(xc, yc, w, h))
                labs.append(NAMES[cid])
            make_contact(
                r['overlay_image'],
                boxes,
                labs,
                CONTACT / f"{source}_{Path(r['overlay_image']).stem}.jpg",
                f"{source} | {r['target_split']} | {r['mapped_class_names']}",
            )

    for _, r in df_synth.head(MAX_SYNTH_CONTACTS).iterrows():
        boxes, labs = [], []
        for cid, xc, yc, w, h in read_yolo_label(r['overlay_label']):
            if cid == 4:
                boxes.append(yolo_xyxy(xc, yc, w, h))
                labs.append('foreign_object:' + r['foreign_subtype'])
        make_contact(
            r['overlay_image'],
            boxes,
            labs,
            CONTACT / f"synthetic_{Path(r['overlay_image']).stem}.jpg",
            f"synthetic relation | {r['foreign_subtype']}",
        )

    print('contacts:', CONTACT)

    # Exact SHA against all V6.2E manifest rows.
    base_hash = {}
    for _, r in manifest.iterrows():
        sha = str(r['sha256'])
        split = str(r['split'])
        base_hash.setdefault(sha, []).append({
            'split': split,
            'image_path_source': str(r['image_path_source']),
        })

    direct_images = [
        Path(x)
        for x in df_direct['overlay_image'].tolist()
        if Path(x).exists()
    ]

    exact = []
    for ip in tqdm(direct_images, desc='direct exact SHA'):
        sha = sha256_file(ip)
        for hit in base_hash.get(sha, []):
            exact.append({
                'external_image': str(ip),
                'base_split': hit['split'],
                'base_image_source': hit['image_path_source'],
                'sha256': sha,
            })

    df_exact = pd.DataFrame(exact)
    df_exact.to_csv(QA / 'external_vs_internal_exact_duplicates.csv', index=False)
    print('Exact external/internal duplicates:', len(df_exact))

    # pHash only against VAL/TEST.
    near = []
    ph_cache_path = CACHE / 'parent_val_test_phash.csv.gz'

    if RUN_PHASH:
        if ph_cache_path.exists():
            df_phbase = pd.read_csv(ph_cache_path)
            print('reuse streamed VAL/TEST pHash cache:', len(df_phbase))
        else:
            ph_rows = []
            subset = manifest[
                manifest['split'].astype(str).isin(PARENT_PHASH_SPLITS)
            ].copy()

            for idx, row in tqdm(
                subset.iterrows(),
                total=len(subset),
                desc='stream parent VAL/TEST pHash',
            ):
                try:
                    raw = read_parent_image_bytes(row)
                    with Image.open(io.BytesIO(raw)) as im:
                        hv = int(str(imagehash.phash(im.convert('RGB'), hash_size=8)), 16)
                    ph_rows.append({
                        'manifest_index': int(idx),
                        'split': str(row['split']),
                        'image_path_source': str(row['image_path_source']),
                        'sha256': str(row['sha256']),
                        'phash_int': hv,
                    })
                except Exception:
                    pass

            df_phbase = pd.DataFrame(ph_rows)
            df_phbase.to_csv(ph_cache_path, index=False, compression='gzip')

        widths = [13, 13, 13, 13, 12]
        shifts = []
        cur = 0
        for w in widths:
            shifts.append(cur)
            cur += w

        buckets = {}
        for idx, r in df_phbase.iterrows():
            hv = int(r['phash_int'])
            for band, (w, sh) in enumerate(zip(widths, shifts)):
                key = (band, (hv >> sh) & ((1 << w) - 1))
                buckets.setdefault(key, []).append(idx)

        for ip in tqdm(direct_images, desc='external vs VAL/TEST pHash'):
            try:
                with Image.open(ip) as im:
                    hv = int(str(imagehash.phash(im.convert('RGB'), hash_size=8)), 16)
            except Exception:
                continue

            cand = set()
            for band, (w, sh) in enumerate(zip(widths, shifts)):
                key = (band, (hv >> sh) & ((1 << w) - 1))
                cand.update(buckets.get(key, []))

            for idx in cand:
                r = df_phbase.loc[idx]
                bh = int(r['phash_int'])
                hd = (hv ^ bh).bit_count()
                if hd <= PHASH_MAX_HAMMING:
                    near.append({
                        'external_image': str(ip),
                        'base_split': r['split'],
                        'base_image_source': r['image_path_source'],
                        'base_sha256': r['sha256'],
                        'phash_hamming': hd,
                    })

    df_near = pd.DataFrame(near)
    df_near.to_csv(QA / 'external_vs_internal_phash_candidates.csv', index=False)
    print('Near-duplicate candidates against VAL/TEST:', len(df_near))

    for h in [BASE_TAR_H, OVERLAY_TAR_H, RECON_ZIP_H]:
        try:
            h.close()
        except Exception:
            pass

    if CLEAN_PARENT_ARCHIVES_AFTER_QA:
        for p in [base_tar_parent, overlay_tar_parent]:
            if Path(p).exists():
                print('Deleting large parent archive after QA:', p)
                Path(p).unlink()

    disk_status()
else:
    print('SKIP dedup rebuild: existing external stage found.')

In [ ]:
# ==============================================================================
# 11. RECOVERY: REVIEW MANIFEST + FREEZE + PACKAGE + UPLOAD
# ==============================================================================
if not STAGE_EXISTS:
    review = []
    for _, r in df_direct.iterrows():
        review.append({
            'candidate_type': 'direct_powerline',
            'overlay_image': r['overlay_image'],
            'overlay_label': r['overlay_label'],
            'source': r['source'],
            'target_split': r['target_split'],
            'mapped_class_names': r['mapped_class_names'],
            'subtype': '',
            'recommended_action': (
                'REVIEW_THEN_KEEP'
                if r['target_split'] == 'train'
                else 'KEEP_AS_EXTERNAL_EVAL'
            ),
            'decision': '',
            'reviewer': '',
            'comment': '',
        })

    for _, r in df_synth.iterrows():
        review.append({
            'candidate_type': 'synthetic_relation',
            'overlay_image': r['overlay_image'],
            'overlay_label': r['overlay_label'],
            'source': r['source'],
            'target_split': 'train',
            'mapped_class_names': 'foreign_object',
            'subtype': r['foreign_subtype'],
            'recommended_action': 'VISUAL_REVIEW_THEN_KEEP',
            'decision': '',
            'reviewer': '',
            'comment': '',
        })

    df_review = pd.DataFrame(review)
    df_review.insert(0, 'review_id', [f'V65R{i:06d}' for i in range(len(df_review))])
    df_review.to_csv(QA / 'v6p5_candidate_review_manifest.csv', index=False)
    display(
        df_review.groupby(
            ['candidate_type', 'source', 'target_split', 'mapped_class_names']
        ).size().reset_index(name='n')
    )

    digest = dir_digest(OVERLAY)
    tag = f'v6p5_ext_{digest[:12]}'

    freeze = {
        'tag': tag,
        'taxonomy': {i: n for i, n in enumerate(NAMES)},
        'base_reference': {
            'dataset': 'V6.2E',
            'tag': V6P2E_TAG,
            'role': 'dedup + internal TRAIN backgrounds only',
        },
        'sources': {
            'OpenImagesV7': {
                'classes': OI_CLASSES,
                'split': 'train',
                'role': 'appearance source for relation-aware synthetic foreign objects',
            },
            'MPCD': {'role': 'broken_strand direct power-line source'},
            'PowerEquipment': {
                'map': POWER_MAP,
                'review_only': POWER_REVIEW_ONLY,
            },
        },
        'counts': {
            'direct_train_images': int((df_direct['target_split'] == 'train').sum()),
            'direct_external_eval_images': int((df_direct['target_split'] == 'external_eval').sum()),
            'synthetic_train_images': int(len(df_synth)),
            'review_candidates': int(len(df_review)),
            'exact_external_internal_duplicates': int(len(df_exact)),
            'phash_candidates': int(len(df_near)),
        },
        'policy': {
            'generic_full_images_direct_merge': False,
            'synthetic_background_split': 'internal train only',
            'dx_sg_direct_merge': False,
            'test_modified': False,
            'auto_promotion': False,
        },
        'overlay_sha256': digest,
    }

    freeze_path = ROOT / f'{tag}.freeze.json'
    freeze_path.write_text(json.dumps(freeze, indent=2, ensure_ascii=False))

    overlay_tar = ROOT / f'{tag}_overlay.tar.gz'
    qa_zip = ROOT / f'{tag}_qa.zip'

    overlay_tar.unlink(missing_ok=True)
    qa_zip.unlink(missing_ok=True)

    with tarfile.open(overlay_tar, 'w:gz') as t:
        t.add(OVERLAY, arcname='v6p5_overlay')

    with zipfile.ZipFile(qa_zip, 'w', compression=zipfile.ZIP_DEFLATED) as z:
        for p in QA.rglob('*'):
            if p.is_file():
                z.write(p, arcname=str(p.relative_to(QA)))

    print('TAG:', tag)
    print('overlay MB=', overlay_tar.stat().st_size / 1024**2)
    print('qa MB=', qa_zip.stat().st_size / 1024**2)

    if UPLOAD_CAMBER:
        rr = f'{STAGE_ROOT}/{tag}'
        for p in [overlay_tar, qa_zip, freeze_path]:
            stash_put(p, f'{rr}/{p.name}')

        # FIXED direct archives expected by existing train notebook.
        stash_put(
            mpcd_stage_tar,
            f'{STAGE_ROOT}/mpcd/mpcd_broken_strand_stage.tar.gz',
        )
        stash_put(
            pe_stage_tar,
            f'{STAGE_ROOT}/power_equipment/power_equipment_stage.tar.gz',
        )

        print('Camber stage root:', rr)
        print('Direct MPCD remote:', f'{STAGE_ROOT}/mpcd/mpcd_broken_strand_stage.tar.gz')
        print('Direct PE remote:', f'{STAGE_ROOT}/power_equipment/power_equipment_stage.tar.gz')

    STAGE_TAG = tag
    STAGE_EXISTS = True
else:
    print('SKIP stage freeze/upload: existing external stage found.')

In [ ]:
# ==============================================================================
# 12. VERIFY / REPAIR DIRECT ARCHIVES FOR EXISTING STAGE
# ==============================================================================
# Existing stage may come from the old staging notebook, which created v6p5_ext_*
# but did NOT necessarily create the two fixed direct archives expected by train.
# We verify them here. If this run just rebuilt staging, they already exist.

def remote_exists(remote):
    r = stash_ls(remote, recursive=False, check=False)
    return r.returncode == 0

mpcd_remote = f'{STAGE_ROOT}/mpcd/mpcd_broken_strand_stage.tar.gz'
pe_remote = f'{STAGE_ROOT}/power_equipment/power_equipment_stage.tar.gz'

mpcd_ok = remote_exists(mpcd_remote)
pe_ok = remote_exists(pe_remote)

print('MPCD direct archive remote exists:', mpcd_ok)
print('PowerEquipment direct archive remote exists:', pe_ok)

if not mpcd_ok or not pe_ok:
    # If old stage exists but direct archives are missing, see if its overlay can be
    # used to reconstruct them without downloading source datasets again.
    print('Direct archives missing. Will reconstruct later from selected stage overlay.')
else:
    print('✅ Direct archives already available for train notebook.')

In [ ]:
# ==============================================================================
# 13. RE-DISCOVER + VERIFY EXTERNAL STAGE
# ==============================================================================
stage_candidates, stage_listing = discover_stage_tags()
print('External stage candidates after recovery:', stage_candidates)

if STAGE_TAG_OVERRIDE:
    STAGE_TAG = STAGE_TAG_OVERRIDE.strip()
elif STAGE_TAG and STAGE_TAG in stage_candidates:
    pass
elif len(stage_candidates) == 1:
    STAGE_TAG = stage_candidates[0]
elif len(stage_candidates) == 0:
    raise RuntimeError('Recovery staging finished but Camber still has no v6p5_ext_* artifact.')
else:
    raise RuntimeError(
        'Có nhiều external stage candidates sau recovery. '
        'Set STAGE_TAG_OVERRIDE bằng tag cần dùng:\n' + '\n'.join(stage_candidates)
    )

STAGE_REMOTE = f'{STAGE_ROOT}/{STAGE_TAG}'
freeze_remote = f'{STAGE_REMOTE}/{STAGE_TAG}.freeze.json'
qa_remote = f'{STAGE_REMOTE}/{STAGE_TAG}_qa.zip'
overlay_remote = f'{STAGE_REMOTE}/{STAGE_TAG}_overlay.tar.gz'

print('SELECTED STAGE_TAG:', STAGE_TAG)
print('STAGE_REMOTE:', STAGE_REMOTE)

# Exact-file verification via download happens in next cell.

In [ ]:
# ==============================================================================
# 14. DOWNLOAD SELECTED STAGE ARTIFACTS
# ==============================================================================
freeze_local = stash_cp(freeze_remote, COMPOSE_CACHE / f'{STAGE_TAG}.freeze.json')
qa_local = stash_cp(qa_remote, COMPOSE_CACHE / f'{STAGE_TAG}_qa.zip')
overlay_local = stash_cp(overlay_remote, COMPOSE_CACHE / f'{STAGE_TAG}_overlay.tar.gz')

stage_freeze = json.loads(freeze_local.read_text())

print('Downloaded selected stage:')
print(' -', freeze_local, f'{freeze_local.stat().st_size / 1024**2:.2f} MB')
print(' -', qa_local, f'{qa_local.stat().st_size / 1024**2:.2f} MB')
print(' -', overlay_local, f'{overlay_local.stat().st_size / 1024**2:.2f} MB')
print(json.dumps(stage_freeze, indent=2, ensure_ascii=False))

In [ ]:
# ==============================================================================
# 15. REPAIR DIRECT ARCHIVES FROM STAGE OVERLAY IF OLD PIPELINE MISSED THEM
# ==============================================================================
mpcd_ok = remote_exists(mpcd_remote)
pe_ok = remote_exists(pe_remote)

if not mpcd_ok or not pe_ok:
    if COMPOSE_EXTRACT.exists():
        shutil.rmtree(COMPOSE_EXTRACT)
    COMPOSE_EXTRACT.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(qa_local) as z:
        z.extractall(COMPOSE_EXTRACT)

    direct_manifest = read_csv_safe(COMPOSE_EXTRACT / 'direct_overlay_manifest.csv')
    if len(direct_manifest) == 0:
        raise RuntimeError(
            'Direct archives are missing and selected stage QA has no direct_overlay_manifest.csv.'
        )

    repair_root = COMPOSE_WORK / 'direct_repair'
    if repair_root.exists():
        shutil.rmtree(repair_root)
    repair_root.mkdir(parents=True, exist_ok=True)

    mpcd_root = repair_root / 'mpcd_broken_strand'
    pe_root = repair_root / 'power_equipment_stage'
    for rr in [mpcd_root, pe_root]:
        for s in ['train', 'external_eval']:
            (rr / 'images' / s).mkdir(parents=True, exist_ok=True)
            (rr / 'labels' / s).mkdir(parents=True, exist_ok=True)

    with tarfile.open(overlay_local, 'r:gz') as t:
        members = {m.name: m for m in t.getmembers() if m.isfile()}
        names = list(members)

        for _, row in direct_manifest.iterrows():
            source = str(row['source'])
            split = str(row['target_split'])
            image_base = Path(str(row['overlay_image'])).name
            label_base = Path(str(row['overlay_label'])).name

            im_name = tar_find_member(names, f'direct/images/{split}/{image_base}')
            lb_name = tar_find_member(names, f'direct/labels/{split}/{label_base}')
            if im_name is None or lb_name is None:
                raise RuntimeError(f'Missing direct overlay pair in stage TAR: {image_base}')

            root = mpcd_root if source == 'MPCD' else pe_root
            (root / 'images' / split / image_base).write_bytes(t.extractfile(members[im_name]).read())
            (root / 'labels' / split / label_base).write_bytes(t.extractfile(members[lb_name]).read())

    repaired_mpcd = COMPOSE_WORK / 'mpcd_broken_strand_stage.tar.gz'
    repaired_pe = COMPOSE_WORK / 'power_equipment_stage.tar.gz'
    repaired_mpcd.unlink(missing_ok=True)
    repaired_pe.unlink(missing_ok=True)

    with tarfile.open(repaired_mpcd, 'w:gz') as t:
        t.add(mpcd_root, arcname='mpcd_broken_strand')
    with tarfile.open(repaired_pe, 'w:gz') as t:
        t.add(pe_root, arcname='power_equipment_stage')

    if not mpcd_ok:
        stash_put(repaired_mpcd, mpcd_remote)
        print('Repaired/uploaded MPCD direct archive.')
    if not pe_ok:
        stash_put(repaired_pe, pe_remote)
        print('Repaired/uploaded PowerEquipment direct archive.')

    assert remote_exists(mpcd_remote), 'MPCD direct archive still missing after repair.'
    assert remote_exists(pe_remote), 'PowerEquipment direct archive still missing after repair.'

print('✅ Direct source archives ready.')

In [ ]:
# ==============================================================================
# 5. EXTRACT QA ONLY + LOAD MANIFESTS
# ==============================================================================
if COMPOSE_EXTRACT.exists(): shutil.rmtree(COMPOSE_EXTRACT)
COMPOSE_EXTRACT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(qa_local) as z:
    z.extractall(COMPOSE_EXTRACT)

review_path = COMPOSE_EXTRACT / "v6p5_candidate_review_manifest.csv"
direct_path = COMPOSE_EXTRACT / "direct_overlay_manifest.csv"
synth_path = COMPOSE_EXTRACT / "synthetic_foreign_manifest.csv"
exact_path = COMPOSE_EXTRACT / "external_vs_internal_exact_duplicates.csv"
phash_path = COMPOSE_EXTRACT / "external_vs_internal_phash_candidates.csv"

required = [review_path, direct_path, synth_path]
missing = [p.name for p in required if not p.exists()]
if missing:
    raise RuntimeError("QA ZIP thiếu file bắt buộc: " + ", ".join(missing))

review = read_csv_safe(review_path)
direct = read_csv_safe(direct_path)
synth = read_csv_safe(synth_path)
exact_dup = read_csv_safe(exact_path)
phash_dup = read_csv_safe(phash_path)

print("review rows:", len(review))
print("direct rows:", len(direct))
print("synthetic rows:", len(synth))
print("exact duplicate rows:", len(exact_dup))
print("pHash candidate rows:", len(phash_dup))
display(review.head(10))

In [ ]:
# ==============================================================================
# 6. HARD QA GATES
# ==============================================================================
EXPECTED_NAMES = [
    "insulator", "insulator_broken", "insulator_flashover",
    "bird_nest", "foreign_object", "broken_strand",
]

taxonomy = stage_freeze.get("taxonomy", {})
if isinstance(taxonomy, dict):
    try: got_names = [taxonomy[k] for k in sorted(taxonomy, key=lambda x: int(x))]
    except Exception: got_names = list(taxonomy.values())
else:
    got_names = list(taxonomy)

print("Taxonomy:", got_names)
if got_names != EXPECTED_NAMES:
    raise RuntimeError(f"Taxonomy mismatch. Expected={EXPECTED_NAMES} Got={got_names}")

policy = stage_freeze.get("policy", {})
if policy.get("test_modified") not in (False, None):
    raise RuntimeError("Hard gate failed: stage freeze says test_modified=True")
if policy.get("auto_promotion") not in (False, None):
    raise RuntimeError("Hard gate failed: staging artifact unexpectedly auto-promoted.")

if len(exact_dup) > 0 and not ALLOW_EXACT_DUPLICATES:
    display(exact_dup.head(50))
    raise RuntimeError(f"Hard gate failed: {len(exact_dup)} exact SHA duplicate(s).")

if len(phash_dup) > 0 and not ALLOW_PHASH_CANDIDATES:
    display(phash_dup.head(50))
    raise RuntimeError(
        f"QA gate: còn {len(phash_dup)} pHash near-duplicate candidate(s). "
        "Review rồi mới set ALLOW_PHASH_CANDIDATES=True."
    )

required_review_cols = {
    "review_id", "candidate_type", "overlay_image", "overlay_label", "source",
    "target_split", "mapped_class_names", "recommended_action", "decision",
}
missing_cols = sorted(required_review_cols - set(review.columns))
if missing_cols:
    raise RuntimeError("Review manifest thiếu cột: " + ", ".join(missing_cols))
if review["review_id"].duplicated().any():
    raise RuntimeError("review_id bị duplicate.")
if review["target_split"].astype(str).str.lower().eq("test").any():
    raise RuntimeError("Có candidate target_split=test. Không được compose vào train.")

print("✅ Hard QA gates passed.")

In [ ]:
# ==============================================================================
# 7. QA SUMMARY
# ==============================================================================
summary_cols = [c for c in [
    "candidate_type", "source", "target_split", "mapped_class_names", "recommended_action"
] if c in review.columns]

if len(review):
    display(review.groupby(summary_cols, dropna=False).size().reset_index(name="n").sort_values("n", ascending=False))

print("Stage freeze counts:")
print(json.dumps(stage_freeze.get("counts", {}), indent=2, ensure_ascii=False))

if len(synth) and "foreign_subtype" in synth.columns:
    display(synth["foreign_subtype"].fillna("<NA>").value_counts().rename_axis("foreign_subtype").reset_index(name="n"))

if len(direct):
    gcols = [c for c in ["source", "target_split", "mapped_class_names"] if c in direct.columns]
    display(direct.groupby(gcols, dropna=False).size().reset_index(name="n").sort_values("n", ascending=False))

In [ ]:
# ==============================================================================
# 8. CONTACT SHEET PREVIEW
# ==============================================================================
from IPython.display import display as ipy_display
from PIL import Image

contact_files = sorted([
    p for p in COMPOSE_EXTRACT.rglob("*")
    if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    and "contact" in str(p.parent).lower()
])
print("Contact sheets found:", len(contact_files))

for p in contact_files[:MAX_CONTACT_SHEETS]:
    print(p.relative_to(COMPOSE_EXTRACT))
    try:
        im = Image.open(p)
        im.thumbnail((1100, 800))
        ipy_display(im)
    except Exception as e:
        print("Could not display:", e)

if len(contact_files) > MAX_CONTACT_SHEETS:
    print(f"... còn {len(contact_files)-MAX_CONTACT_SHEETS} contact sheets.")

In [ ]:
# ==============================================================================
# 9. RESOLVE KEEP / DROP
# ==============================================================================
resolved = review.copy()
resolved["decision"] = resolved["decision"].map(normalize_decision)

for rid, decision in MANUAL_DECISIONS.items():
    decision = normalize_decision(decision)
    if decision not in {"KEEP", "DROP"}:
        raise ValueError(f"Invalid MANUAL_DECISIONS[{rid!r}]={decision!r}")
    mask = resolved["review_id"].astype(str) == str(rid)
    if not mask.any(): raise KeyError(f"review_id not found: {rid}")
    resolved.loc[mask, "decision"] = decision

if AUTO_APPROVE_RECOMMENDED:
    blank = resolved["decision"].eq("")
    train = resolved["target_split"].astype(str).str.lower().eq("train")
    recommended_keep = resolved["recommended_action"].astype(str).str.upper().str.contains("KEEP", na=False)
    resolved.loc[blank & train & recommended_keep, "decision"] = "KEEP"
    resolved.loc[blank & ~(train & recommended_keep), "decision"] = "DROP"

pending_train = resolved[
    resolved["target_split"].astype(str).str.lower().eq("train") & resolved["decision"].eq("")
].copy()

display(
    resolved.assign(decision_display=resolved["decision"].replace("", "PENDING"))
    .groupby(["candidate_type", "target_split", "decision_display"], dropna=False)
    .size().reset_index(name="n")
)

if len(pending_train):
    cols = [c for c in ["review_id", "candidate_type", "source", "mapped_class_names", "subtype", "recommended_action"] if c in pending_train.columns]
    display(pending_train[cols].head(100))
    raise RuntimeError(
        f"Còn {len(pending_train)} TRAIN candidate(s) chưa có decision.\n"
        "Xem contact sheets, điền MANUAL_DECISIONS hoặc sau khi review set AUTO_APPROVE_RECOMMENDED=True, rồi Run All lại."
    )

resolved.loc[resolved["target_split"].astype(str).str.lower().ne("train"), "decision"] = "DROP"
resolved_path = COMPOSE_OUT / "v6p5_candidate_review_manifest.resolved.csv"
resolved.to_csv(resolved_path, index=False)
print("✅ Review decisions resolved:", resolved_path)

In [ ]:
# ==============================================================================
# 10. BUILD APPROVED SYNTHETIC TRAIN ARCHIVE — STREAMING / LOW DISK
# ==============================================================================
keep_synth = resolved[
    resolved["candidate_type"].astype(str).eq("synthetic_relation")
    & resolved["target_split"].astype(str).str.lower().eq("train")
    & resolved["decision"].eq("KEEP")
].copy()

print("Approved synthetic rows:", len(keep_synth))
if len(keep_synth) == 0:
    raise RuntimeError("Không có synthetic_relation TRAIN candidate nào được KEEP.")

tmp_synth_tar = COMPOSE_OUT / "approved_synthetic_train.tar.gz"

with tarfile.open(overlay_local, "r:gz") as src:
    members = {m.name: m for m in src.getmembers() if m.isfile()}
    names = list(members)
    with tarfile.open(tmp_synth_tar, "w:gz") as dst:
        included = []
        for _, row in keep_synth.iterrows():
            image_base = Path(str(row["overlay_image"])).name
            label_base = Path(str(row["overlay_label"])).name
            image_suffix = f"synthetic_foreign/images/train/{image_base}"
            label_suffix = f"synthetic_foreign/labels/train/{label_base}"
            im_name = tar_find_member(names, image_suffix)
            lb_name = tar_find_member(names, label_suffix)
            if im_name is None: raise RuntimeError(f"Missing synthetic image: {image_suffix}")
            if lb_name is None: raise RuntimeError(f"Missing synthetic label: {label_suffix}")
            im_member, lb_member = members[im_name], members[lb_name]
            im_raw = src.extractfile(im_member).read()
            lb_raw = src.extractfile(lb_member).read()
            out_im = f"v6p5_overlay/synthetic_foreign/images/train/{image_base}"
            out_lb = f"v6p5_overlay/synthetic_foreign/labels/train/{label_base}"
            add_bytes_to_tar(dst, out_im, im_raw, im_member)
            add_bytes_to_tar(dst, out_lb, lb_raw, lb_member)
            included.append({
                "review_id": row["review_id"], "image": out_im, "label": out_lb,
                "image_sha256": hashlib.sha256(im_raw).hexdigest(),
                "label_sha256": hashlib.sha256(lb_raw).hexdigest(),
            })

included_df = pd.DataFrame(included)
if len(included_df) != len(keep_synth):
    raise RuntimeError(f"Approved synthetic mismatch: {len(included_df)} != {len(keep_synth)}")

approved_manifest_path = COMPOSE_OUT / "approved_synthetic_manifest.csv"
included_df.to_csv(approved_manifest_path, index=False)
print("Archive:", tmp_synth_tar)
print("Archive MB:", round(tmp_synth_tar.stat().st_size / 1024**2, 3))
print("SHA256:", sha256_file(tmp_synth_tar))
display(included_df.head())

In [ ]:
# ==============================================================================
# 11. FINAL COMPOSE QA + IMMUTABLE TAG
# ==============================================================================
# Train launcher hiện tại lấy direct TRAIN rows trực tiếp từ STAGE_ROOT/mpcd và
# STAGE_ROOT/power_equipment. Compose artifact gate QA/review và cung cấp filtered
# synthetic archive tương thích với launcher đó.

train_rows = resolved[resolved["target_split"].astype(str).str.lower().eq("train")].copy()
kept = train_rows[train_rows["decision"].eq("KEEP")].copy()
dropped = train_rows[train_rows["decision"].eq("DROP")].copy()

review_digest_payload = [{
    "review_id": str(r["review_id"]),
    "candidate_type": str(r["candidate_type"]),
    "source": str(r["source"]),
    "mapped_class_names": str(r["mapped_class_names"]),
    "subtype": str(r.get("subtype", "")),
    "decision": str(r["decision"]),
} for _, r in resolved.sort_values("review_id").iterrows()]

compose_seed = {
    "source_stage_tag": STAGE_TAG,
    "source_overlay_sha256": stage_freeze.get("overlay_sha256"),
    "review_decisions_sha256": stable_json_hash(review_digest_payload),
    "approved_synthetic_archive_sha256": sha256_file(tmp_synth_tar),
    "taxonomy": EXPECTED_NAMES,
}

compose_digest = stable_json_hash(compose_seed)
COMPOSE_TAG = f"v6p5_compose_{compose_digest[:12]}"
COMPOSE_REMOTE = f"{COMPOSE_ROOT}/{COMPOSE_TAG}"

synthetic_final = COMPOSE_OUT / f"{COMPOSE_TAG}_synthetic_approved.tar.gz"
shutil.copy2(tmp_synth_tar, synthetic_final)

qa_summary = {
    "compose_tag": COMPOSE_TAG,
    "source_stage_tag": STAGE_TAG,
    "review_total": int(len(resolved)),
    "train_candidates": int(len(train_rows)),
    "train_keep": int(len(kept)),
    "train_drop": int(len(dropped)),
    "approved_synthetic": int(len(keep_synth)),
    "exact_duplicate_candidates": int(len(exact_dup)),
    "phash_candidates": int(len(phash_dup)),
    "allow_phash_candidates": bool(ALLOW_PHASH_CANDIDATES),
    "auto_approve_recommended": bool(AUTO_APPROVE_RECOMMENDED),
}
qa_summary_path = COMPOSE_OUT / f"{COMPOSE_TAG}.qa_summary.json"
qa_summary_path.write_text(json.dumps(qa_summary, indent=2, ensure_ascii=False))
print("COMPOSE_TAG:", COMPOSE_TAG)
print(json.dumps(qa_summary, indent=2, ensure_ascii=False))

In [ ]:
# ==============================================================================
# 12. WRITE INDEX + FREEZE
# ==============================================================================
synthetic_remote = f"{COMPOSE_REMOTE}/{synthetic_final.name}"
resolved_remote = f"{COMPOSE_REMOTE}/{COMPOSE_TAG}.review.resolved.csv"
approved_manifest_remote = f"{COMPOSE_REMOTE}/{COMPOSE_TAG}.synthetic_manifest.csv"
qa_summary_remote = f"{COMPOSE_REMOTE}/{COMPOSE_TAG}.qa_summary.json"

compose_index = {
    "compose_tag": COMPOSE_TAG,
    "source_stage_tag": STAGE_TAG,
    "source_stage_remote": STAGE_REMOTE,
    # IMPORTANT: train launcher đọc field này.
    "synthetic_archive": synthetic_remote,
    "resolved_review_manifest": resolved_remote,
    "approved_synthetic_manifest": approved_manifest_remote,
    "qa_summary": qa_summary_remote,
    "source_stage_freeze": freeze_remote,
    "source_stage_qa": qa_remote,
    "source_stage_overlay": overlay_remote,
    "direct_archives": {
        "mpcd": f"{STAGE_ROOT}/mpcd/mpcd_broken_strand_stage.tar.gz",
        "power_equipment": f"{STAGE_ROOT}/power_equipment/power_equipment_stage.tar.gz",
    },
}

index_path = COMPOSE_OUT / f"{COMPOSE_TAG}.index.json"
index_path.write_text(json.dumps(compose_index, indent=2, ensure_ascii=False))

compose_freeze = {
    "tag": COMPOSE_TAG,
    "type": "v6p5_compose_qa",
    "source_stage_tag": STAGE_TAG,
    "source_stage_freeze": stage_freeze,
    "taxonomy": {i: n for i, n in enumerate(EXPECTED_NAMES)},
    "counts": qa_summary,
    "policy": {
        "test_modified": False,
        "val_modified": False,
        "external_eval_in_training": False,
        "synthetic_train_only": True,
        "review_resolved": True,
        "auto_approve_recommended": bool(AUTO_APPROVE_RECOMMENDED),
        "phash_candidates_explicitly_allowed": bool(ALLOW_PHASH_CANDIDATES),
    },
    "artifacts": {
        "synthetic_archive": {"remote": synthetic_remote, "sha256": sha256_file(synthetic_final)},
        "resolved_review_manifest": {"remote": resolved_remote, "sha256": sha256_file(resolved_path)},
        "approved_synthetic_manifest": {"remote": approved_manifest_remote, "sha256": sha256_file(approved_manifest_path)},
        "qa_summary": {"remote": qa_summary_remote, "sha256": sha256_file(qa_summary_path)},
        "index": {"remote": f"{COMPOSE_REMOTE}/{index_path.name}", "sha256": sha256_file(index_path)},
    },
    "compose_seed": compose_seed,
}

freeze_out = COMPOSE_OUT / f"{COMPOSE_TAG}.freeze.json"
freeze_out.write_text(json.dumps(compose_freeze, indent=2, ensure_ascii=False))
print(json.dumps(compose_index, indent=2, ensure_ascii=False))
print("Freeze:", freeze_out)

In [ ]:
# ==============================================================================
# 13. VERIFY LOCAL COMPOSE ARTIFACTS
# ==============================================================================
with tarfile.open(synthetic_final, "r:gz") as t:
    file_names = [m.name for m in t.getmembers() if m.isfile()]

img_names = sorted([n for n in file_names if "synthetic_foreign/images/train/" in n and Path(n).suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}])
lbl_names = sorted([n for n in file_names if "synthetic_foreign/labels/train/" in n and n.lower().endswith(".txt")])
img_stems = {Path(n).stem for n in img_names}
lbl_stems = {Path(n).stem for n in lbl_names}

print("Synthetic images:", len(img_names))
print("Synthetic labels:", len(lbl_names))
if img_stems - lbl_stems: raise RuntimeError(f"Missing labels: {sorted(img_stems-lbl_stems)[:20]}")
if lbl_stems - img_stems: raise RuntimeError(f"Orphan labels: {sorted(lbl_stems-img_stems)[:20]}")
if len(img_names) != len(keep_synth):
    raise RuntimeError(f"Archive count mismatch: {len(img_names)} != {len(keep_synth)}")

with tarfile.open(synthetic_final, "r:gz") as t:
    for m in t.getmembers():
        if not (m.isfile() and m.name.lower().endswith(".txt")): continue
        txt = t.extractfile(m).read().decode("utf-8", errors="replace")
        for ln_no, line in enumerate(txt.splitlines(), 1):
            line = line.strip()
            if not line: continue
            parts = line.split()
            if len(parts) != 5: raise RuntimeError(f"Bad YOLO row {m.name}:{ln_no}: {line}")
            cid = int(float(parts[0])); vals = list(map(float, parts[1:]))
            if not (0 <= cid < len(EXPECTED_NAMES)): raise RuntimeError(f"Bad class id {cid} at {m.name}:{ln_no}")
            if not all(np.isfinite(vals)): raise RuntimeError(f"Non-finite bbox at {m.name}:{ln_no}")
            if not all(0.0 <= v <= 1.0 for v in vals): raise RuntimeError(f"Out-of-bound bbox at {m.name}:{ln_no}: {vals}")

print("✅ Local compose archive integrity passed.")

In [ ]:
# ==============================================================================
# 14. UPLOAD COMPOSE CANDIDATE TO CAMBER
# ==============================================================================
existing = stash_ls(COMPOSE_ROOT, recursive=True, check=False)
existing_tags = extract_tag_tokens(existing.stdout, "v6p5_compose_") if existing.returncode == 0 else []
print("Existing compose tags:", existing_tags)

if COMPOSE_TAG in existing_tags and not ALLOW_EXISTING_COMPOSE_TAG:
    raise RuntimeError(
        f"{COMPOSE_TAG} đã tồn tại trên Camber. Nếu muốn overwrite thật sự, "
        "set ALLOW_EXISTING_COMPOSE_TAG=True."
    )

if UPLOAD_CAMBER:
    uploads = [
        (synthetic_final, synthetic_remote),
        (resolved_path, resolved_remote),
        (approved_manifest_path, approved_manifest_remote),
        (qa_summary_path, qa_summary_remote),
        (index_path, f"{COMPOSE_REMOTE}/{index_path.name}"),
        (freeze_out, f"{COMPOSE_REMOTE}/{freeze_out.name}"),
    ]
    for local, remote in uploads:
        print("UPLOAD:", local.name, "->", remote)
        stash_put(local, remote)
    print("✅ Upload complete.")
else:
    print("UPLOAD_CAMBER=False; local artifacts only.")

In [ ]:
# ==============================================================================
# 15. SERVER-SIDE VERIFY — WHAT TRAIN NOTEBOOK WILL SEE
# ==============================================================================
r = stash_ls(COMPOSE_REMOTE, recursive=True)
print(r.stdout)
server_listing = r.stdout

must_appear = [f"{COMPOSE_TAG}.index.json", f"{COMPOSE_TAG}.freeze.json", synthetic_final.name]
missing_server = [x for x in must_appear if x not in server_listing]
if missing_server:
    raise RuntimeError("Upload verify failed: " + ", ".join(missing_server))

root_listing = stash_ls(COMPOSE_ROOT, recursive=True).stdout
compose_candidates = extract_tag_tokens(root_listing, "v6p5_compose_")
print("Compose candidates visible to train launcher:", compose_candidates)
if COMPOSE_TAG not in compose_candidates:
    raise RuntimeError("Compose tag uploaded nhưng auto-discovery ở COMPOSE_ROOT chưa thấy tag.")

print("\n" + "=" * 78)
print("READY FOR DATA ABLATION TRAIN")
print("=" * 78)
print("COMPOSE_TAG =", COMPOSE_TAG)
print("COMPOSE_REMOTE =", COMPOSE_REMOTE)
print("\nMở KAGGLE_V6P5_CAMBER_YOLO11N_640_DATA_ABLATION_TRAIN")
print("Giữ COMPOSE_TAG_OVERRIDE = None")
print(f"Expected: Compose candidates: ['{COMPOSE_TAG}']")
print("=" * 78)

## Sau khi cell cuối thành công

Bạn **không đổi** config của notebook train.

Trong `KAGGLE_V6P5_CAMBER_YOLO11N_640_DATA_ABLATION_TRAIN.ipynb` giữ:

```python
COMPOSE_TAG_OVERRIDE = None
```

Cell discovery phải nhìn thấy:

```text
Compose candidates: ['v6p5_compose_xxxxxxxxxxxx']
```

Ngoài compose tag, notebook này cũng đảm bảo hai artifact train đang hardcode tồn tại:

```text
stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage/mpcd/mpcd_broken_strand_stage.tar.gz
stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage/power_equipment/power_equipment_stage.tar.gz
```

Như vậy không còn cái cảnh sửa được lỗi A rồi chạy thêm ba cell gặp lỗi B, một phong tục khá được notebook ML ưa chuộng.